# Ablation - FLAN-T5-Larg

In [ ]:
!pip -q install -U transformers accelerate sentencepiece
!pip -q install datasets==3.6.0 evaluate nltk rouge_score

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
print('cwd:', os.getcwd())

In [ ]:
import os
import json
import time
import random

import numpy as np
import torch

from datasets import load_dataset, DatasetDict
import evaluate
import nltk

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel

nltk.download('wordnet')
nltk.download('punkt')
nltk.download('omw-1.4')

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive'
STATE_PATH = os.path.join(DRIVE_DIR, 'ablation_flan_t5_large_student_v2_self_instruct_config.json')

def default_state():
    return {
        'config': {
            'seed': 42,
            'dataset_id': 'yizhongw/self_instruct',
            'model_id': 'google/flan-t5-large',
            'similarity_model_id': 'Qwen/Qwen3-Embedding-0.6B',
            'dataset_type': 'test',
            'max_num_samples': 128,
            'batch_size': 64,
            'tokenizer_max_length': 128,
            'max_new_tokens': 128,
            'do_sample': False,
            'num_beams': 1,
            'sweep_metric_names': ['similarity'],
            'full_metric_names': ['similarity', 'rouge', 'meteor'],
            'debug_metric_names': ['rouge', 'meteor'],
            'compute_per_sample_meteor': True,
            'similarity_threshold': 0.5,
            'rewrite_existing_metrics': False
        },
        'completed_keys': [],
        'results': {},
        'meta': {
            'last_saved_unix': None
        }
    }

def _safe_merge(default_obj, loaded_obj):
    if not isinstance(default_obj, dict):
        return loaded_obj
    merged = dict(default_obj)
    if not isinstance(loaded_obj, dict):
        return merged
    for key, value in loaded_obj.items():
        if key in merged and isinstance(merged[key], dict):
            merged[key] = _safe_merge(merged[key], value)
        else:
            merged[key] = value
    return merged

def load_state(state_path):
    defaults = default_state()
    if not os.path.exists(state_path):
        return defaults
    try:
        with open(state_path, 'r') as f:
            loaded = json.load(f)
        state = _safe_merge(defaults, loaded)
    except Exception as error:
        print(f'Warning: failed loading state at {state_path}: {error}')
        print('Using default state.')
        state = defaults

    if not isinstance(state.get('completed_keys'), list):
        state['completed_keys'] = []
    if not isinstance(state.get('results'), dict):
        state['results'] = {}

    return state

def save_state(state_path, state):
    os.makedirs(os.path.dirname(state_path), exist_ok=True)
    state['meta']['last_saved_unix'] = time.time()

    tmp_path = state_path + '.tmp'
    with open(tmp_path, 'w') as f:
        json.dump(state, f)
    os.replace(tmp_path, state_path)

STATE = load_state(STATE_PATH)
CONFIG = STATE['config']

FORCE_FAST_SWEEP_CONFIG = False
if FORCE_FAST_SWEEP_CONFIG:
    CONFIG['max_num_samples'] = 128
    CONFIG['batch_size'] = 64
    CONFIG['max_new_tokens'] = 128
    CONFIG['tokenizer_max_length'] = 128
    CONFIG['do_sample'] = False
    CONFIG['num_beams'] = 1
    CONFIG['sweep_metric_names'] = ['similarity']
    CONFIG['debug_metric_names'] = ['rouge', 'meteor']
    save_state(STATE_PATH, STATE)

print('State path:', STATE_PATH)
print('Completed keys:', len(STATE['completed_keys']))
print('Active config:', CONFIG)

In [ ]:
SEED = int(CONFIG['seed'])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float32

print('Device:', DEVICE)
print('Dtype:', DTYPE)

In [ ]:
import os
import torch
from transformers import AutoConfig, AutoModelForSeq2SeqLM, AutoTokenizer, AutoModel

# --- Custom Model Configuration ---
USE_CUSTOM_DISTILLED_MODEL = True
# Updated: Points directly to the 'final' folder as specified
CUSTOM_MODEL_PATH = '/content/drive/MyDrive/final'
# -------------------------------

MODEL_ID = CUSTOM_MODEL_PATH if USE_CUSTOM_DISTILLED_MODEL else CONFIG['model_id']
SIMILARITY_MODEL_ID = CONFIG['similarity_model_id']

print(f"Loading model from: {MODEL_ID}")

# Use local tokenizer if available, else fallback to base FLAN-T5-Large tokenizer
has_local_tokenizer = os.path.exists(os.path.join(CUSTOM_MODEL_PATH, 'tokenizer.json'))
tokenizer_id = MODEL_ID if (USE_CUSTOM_DISTILLED_MODEL and has_local_tokenizer) else "google/flan-t5-large"
flan_tokenizer = AutoTokenizer.from_pretrained(tokenizer_id, use_fast=False)

# Load Model
if USE_CUSTOM_DISTILLED_MODEL:
    print("Loading custom distilled model using standard from_pretrained...")
    # Note: If SafetensorError persists, the file in Drive is corrupted/incomplete.
    flan_model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_ID,
        dtype=DTYPE,
        low_cpu_mem_usage=True
    )
else:
    flan_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, dtype=DTYPE, low_cpu_mem_usage=True)

flan_model = flan_model.to(DEVICE)
flan_model.eval()

# Load Similarity Model for Evaluation
similarity_tokenizer = AutoTokenizer.from_pretrained(SIMILARITY_MODEL_ID)
similarity_model = AutoModel.from_pretrained(SIMILARITY_MODEL_ID)
similarity_model = similarity_model.to('cpu')
similarity_model.eval()

print('\n' + '='*30)
print(f'STATUS: {MODEL_ID} READY')
print('='*30)

In [ ]:
def preprocess_self_instruct_dataset(dataset, seed):
    def preprocess_sample(sample):
        prompt = sample.get('prompt', '').strip()
        completion = sample.get('completion', '').strip()
        sample['input'] = ' '.join(prompt.split())
        sample['output'] = completion
        return sample

    dataset = dataset.map(preprocess_sample)
    dataset = dataset.remove_columns(['prompt', 'completion'])

    train_weight = 56000
    validation_weight = 8000
    test_weight = 16000

    split_0 = dataset['train'].train_test_split(
        train_size=train_weight / (train_weight + validation_weight + test_weight),
        test_size=(validation_weight + test_weight) / (train_weight + validation_weight + test_weight),
        seed=seed
    )

    split_1 = split_0['test'].train_test_split(
        train_size=validation_weight / (validation_weight + test_weight),
        test_size=test_weight / (validation_weight + test_weight),
        seed=seed
    )

    return DatasetDict({
        'train': split_0['train'],
        'validation': split_1['train'],
        'test': split_1['test']
    })

self_instruct = load_dataset(CONFIG['dataset_id'], trust_remote_code=True)
self_instruct = preprocess_self_instruct_dataset(self_instruct, SEED)

dataset_type = CONFIG['dataset_type']
max_num_samples = int(CONFIG['max_num_samples'])
self_instruct_subset = self_instruct[dataset_type].select(range(min(len(self_instruct[dataset_type]), max_num_samples)))

print('Subset size:', len(self_instruct_subset))
print('Sample input:', self_instruct_subset[0]['input'][:150])

In [ ]:
def drop_encoder_block(model, encoder_block_index):
    encoder_block = model.encoder.block[encoder_block_index]

    def hook_function(module, inputs, outputs):
        return (inputs[0],) + outputs[1:]

    return encoder_block.register_forward_hook(hook_function)


def drop_decoder_block(model, decoder_block_index):
    decoder_block = model.decoder.block[decoder_block_index]

    def hook_function(module, inputs, outputs):
        return (inputs[0],) + outputs[1:]

    return decoder_block.register_forward_hook(hook_function)


def drop_encoder_block_feed_forward_layer(model, encoder_block_index):
    feed_forward_layer = model.encoder.block[encoder_block_index].layer[1]

    def hook_function(module, inputs, outputs):
        return inputs[0]

    return feed_forward_layer.register_forward_hook(hook_function)


def drop_decoder_block_feed_forward_layer(model, decoder_block_index):
    feed_forward_layer = model.decoder.block[decoder_block_index].layer[2]

    def hook_function(module, inputs, outputs):
        return inputs[0]

    return feed_forward_layer.register_forward_hook(hook_function)


def mask_encoder_block_feed_forward_layer(model, encoder_block_index, mask_threshold):
    feed_forward_layer = model.encoder.block[encoder_block_index].layer[1]

    def hook_function(module, inputs, outputs):
        outputs_mask = (torch.rand_like(outputs) > mask_threshold).float()
        return outputs * outputs_mask

    return feed_forward_layer.register_forward_hook(hook_function)


def mask_decoder_block_feed_forward_layer(model, decoder_block_index, mask_threshold):
    feed_forward_layer = model.decoder.block[decoder_block_index].layer[2]

    def hook_function(module, inputs, outputs):
        outputs_mask = (torch.rand_like(outputs) > mask_threshold).float()
        return outputs * outputs_mask

    return feed_forward_layer.register_forward_hook(hook_function)


def add_gaussian_noise_encoder_block(model, encoder_block_index, standard_deviation=0.1, mean=0.0):
    encoder_block = model.encoder.block[encoder_block_index]

    def hook_function(module, inputs, outputs):
        hidden_activations = outputs[0]
        outputs_noise = mean + standard_deviation * torch.randn_like(hidden_activations)
        return (hidden_activations + outputs_noise,) + outputs[1:]

    return encoder_block.register_forward_hook(hook_function)


def add_gaussian_noise_decoder_block(model, decoder_block_index, standard_deviation=0.1, mean=0.0):
    decoder_block = model.decoder.block[decoder_block_index]

    def hook_function(module, inputs, outputs):
        hidden_activations = outputs[0]
        outputs_noise = mean + standard_deviation * torch.randn_like(hidden_activations)
        return (hidden_activations + outputs_noise,) + outputs[1:]

    return decoder_block.register_forward_hook(hook_function)


def add_gaussian_noise_encoder_block_feed_forward_layer(model, encoder_block_index, standard_deviation=0.1, mean=0.0):
    feed_forward_layer = model.encoder.block[encoder_block_index].layer[1]

    def hook_function(module, inputs, outputs):
        outputs_noise = mean + standard_deviation * torch.randn_like(outputs)
        return outputs + outputs_noise

    return feed_forward_layer.register_forward_hook(hook_function)


def add_gaussian_noise_decoder_block_feed_forward_layer(model, decoder_block_index, standard_deviation=0.1, mean=0.0):
    feed_forward_layer = model.decoder.block[decoder_block_index].layer[2]

    def hook_function(module, inputs, outputs):
        outputs_noise = mean + standard_deviation * torch.randn_like(outputs)
        return outputs + outputs_noise

    return feed_forward_layer.register_forward_hook(hook_function)


def _first_tensor(outputs):
    return outputs[0] if isinstance(outputs, tuple) else outputs


def _replace_first_tensor(outputs, new_first):
    return (new_first,) + outputs[1:] if isinstance(outputs, tuple) else new_first


def mask_self_attention_heads_encoder_block(model, encoder_block_index, mask_threshold):
    self_attention_layer = model.encoder.block[encoder_block_index].layer[0].SelfAttention
    num_heads = self_attention_layer.n_heads
    # Fix: Derive head dimension from inner_dim instead of key_dim
    head_dimension = self_attention_layer.inner_dim // num_heads

    def hook_function(module, inputs, outputs):
        hidden = _first_tensor(outputs)
        batch_size, sequence_length, d_model = hidden.shape
        if d_model != num_heads * head_dimension:
            return outputs
        attention_outputs = hidden.view(batch_size, sequence_length, num_heads, head_dimension)
        outputs_mask = (torch.rand(num_heads, device=hidden.device) > mask_threshold).float()
        masked_outputs = attention_outputs * outputs_mask.view(1, 1, num_heads, 1)
        return _replace_first_tensor(outputs, masked_outputs.view(batch_size, sequence_length, d_model))

    return self_attention_layer.register_forward_hook(hook_function)


def mask_self_attention_heads_decoder_block(model, decoder_block_index, mask_threshold):
    self_attention_layer = model.decoder.block[decoder_block_index].layer[0].SelfAttention
    num_heads = self_attention_layer.n_heads
    # Fix: Derive head dimension from inner_dim instead of key_dim
    head_dimension = self_attention_layer.inner_dim // num_heads

    def hook_function(module, inputs, outputs):
        hidden = _first_tensor(outputs)
        batch_size, sequence_length, d_model = hidden.shape
        if d_model != num_heads * head_dimension:
            return outputs
        attention_outputs = hidden.view(batch_size, sequence_length, num_heads, head_dimension)
        outputs_mask = (torch.rand(num_heads, device=hidden.device) > mask_threshold).float()
        masked_outputs = attention_outputs * outputs_mask.view(1, 1, num_heads, 1)
        return _replace_first_tensor(outputs, masked_outputs.view(batch_size, sequence_length, d_model))

    return self_attention_layer.register_forward_hook(hook_function)


def mask_cross_attention_heads_decoder_block(model, decoder_block_index, mask_threshold):
    cross_attention_layer = model.decoder.block[decoder_block_index].layer[1].EncDecAttention
    num_heads = cross_attention_layer.n_heads
    # Fix: Derive head dimension from inner_dim instead of key_dim
    head_dimension = cross_attention_layer.inner_dim // num_heads

    def hook_function(module, inputs, outputs):
        hidden = _first_tensor(outputs)
        batch_size, sequence_length, d_model = hidden.shape
        if d_model != num_heads * head_dimension:
            return outputs
        attention_outputs = hidden.view(batch_size, sequence_length, num_heads, head_dimension)
        outputs_mask = (torch.rand(num_heads, device=hidden.device) > mask_threshold).float()
        masked_outputs = attention_outputs * outputs_mask.view(1, 1, num_heads, 1)
        return _replace_first_tensor(outputs, masked_outputs.view(batch_size, sequence_length, d_model))

    return cross_attention_layer.register_forward_hook(hook_function)


def obtain_model_generated_outputs_subset(model, tokenizer, dataset_subset):
    def collate_function(batch):
        input_texts = [sample['input'] for sample in batch]
        return tokenizer(
            input_texts,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=int(CONFIG['tokenizer_max_length'])
        )

    tokenizer.padding_side = 'right'

    data_loader = torch.utils.data.DataLoader(
        dataset_subset,
        batch_size=int(CONFIG['batch_size']),
        shuffle=False,
        collate_fn=collate_function
    )

    generated_outputs = []

    gen_device = DEVICE
    model.to(gen_device)
    model.eval()

    for batch_index, batch in enumerate(data_loader):
        batch = {key: value.to(gen_device) for key, value in batch.items()}

        with torch.no_grad():
            output_tokens = model.generate(
                **batch,
                max_new_tokens=int(CONFIG['max_new_tokens']),
                do_sample=bool(CONFIG['do_sample']),
                num_beams=int(CONFIG['num_beams'])
            )

        decoded_outputs = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)
        generated_outputs.extend(decoded_outputs)

        print(f'Generation batch {batch_index + 1}/{len(data_loader)} finished')

    if 'generated_output' in dataset_subset.column_names:
        dataset_subset = dataset_subset.remove_columns('generated_output')
    dataset_subset = dataset_subset.add_column('generated_output', generated_outputs)

    return dataset_subset


ROUGE_METRIC = None
METEOR_METRIC = None

def evaluate_model_generative_task(model, tokenizer, dataset_subset, metric_names=None):
    global ROUGE_METRIC
    global METEOR_METRIC

    if metric_names is None:
        metric_names = ['similarity']

    outputs = dataset_subset['output']
    generated_outputs = dataset_subset['generated_output']

    tokenizer.padding_side = 'right'
    batch_size = int(CONFIG['batch_size'])
    similarities = []
    num_samples = len(outputs)

    eval_device = DEVICE if torch.cuda.is_available() else 'cpu'
    model.to(eval_device)
    model.eval()

    for index in range(0, num_samples, batch_size):
        batch_outputs = outputs[index:min(index + batch_size, num_samples)]
        batch_generated_outputs = generated_outputs[index:min(index + batch_size, num_samples)]

        tokenized_outputs = tokenizer(
            batch_outputs,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=int(CONFIG['tokenizer_max_length'])
        ).to(eval_device)

        tokenized_generated_outputs = tokenizer(
            batch_generated_outputs,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=int(CONFIG['tokenizer_max_length'])
        ).to(eval_device)

        with torch.no_grad():
            model_outputs = model(**tokenized_outputs)
            model_generated_outputs = model(**tokenized_generated_outputs)

            last_token_index_outputs = tokenized_outputs['attention_mask'].sum(dim=1) - 1
            last_token_index_generated = tokenized_generated_outputs['attention_mask'].sum(dim=1) - 1

            embedded_outputs = model_outputs.last_hidden_state[
                torch.arange(last_token_index_outputs.shape[0]), last_token_index_outputs
            ]
            embedded_generated_outputs = model_generated_outputs.last_hidden_state[
                torch.arange(last_token_index_generated.shape[0]), last_token_index_generated
            ]

            embedded_outputs = torch.nn.functional.normalize(embedded_outputs, p=2, dim=1)
            embedded_generated_outputs = torch.nn.functional.normalize(embedded_generated_outputs, p=2, dim=1)

            batch_similarities = (embedded_outputs * embedded_generated_outputs).sum(dim=1).cpu().tolist()
            similarities.extend(batch_similarities)

        print(f'Similarity samples processed: {min(index + batch_size, num_samples)}/{num_samples}')

    model.to(DEVICE if torch.cuda.is_available() else 'cpu')
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    results = {}

    if 'similarity' in metric_names:
        results['similarities'] = similarities
        results['similarity_mean'] = float(sum(similarities) / len(similarities))
        threshold = float(CONFIG.get('similarity_threshold', 0.5))
        results['similarity_hard_predictions'] = [1 if score >= threshold else 0 for score in similarities]
        results['similarity_hard_prediction_mean'] = float(sum(results['similarity_hard_predictions']) / len(results['similarity_hard_predictions']))

    if 'rouge' in metric_names:
        if ROUGE_METRIC is None:
            ROUGE_METRIC = evaluate.load('rouge')
        results['rouge'] = ROUGE_METRIC.compute(predictions=generated_outputs, references=outputs)

    if 'meteor' in metric_names:
        if METEOR_METRIC is None:
            METEOR_METRIC = evaluate.load('meteor')
        results['meteor_mean'] = METEOR_METRIC.compute(predictions=generated_outputs, references=outputs)['meteor']
        if CONFIG.get('compute_per_sample_meteor', False):
            results['meteor'] = []
            for sample_index in range(num_samples):
                sample_meteor = METEOR_METRIC.compute(
                    predictions=[generated_outputs[sample_index]],
                    references=[outputs[sample_index]]
                )['meteor']
                results['meteor'].append(sample_meteor)

    return results

In [ ]:
NUM_ENCODER_BLOCKS = len(flan_model.encoder.block)
NUM_DECODER_BLOCKS = len(flan_model.decoder.block)

SWEEP_METRIC_NAMES = CONFIG.get('sweep_metric_names', ['similarity'])

def run_with_hooks(experiment_key, register_hooks_fn, metric_names=None):
    if experiment_key in STATE['completed_keys']:
        print(f'Skipping {experiment_key} (already completed)')
        return

    if metric_names is None:
        metric_names = SWEEP_METRIC_NAMES

    hooks = []
    started_at = time.time()
    try:
        hooks = register_hooks_fn()
        eval_subset = obtain_model_generated_outputs_subset(flan_model, flan_tokenizer, self_instruct_subset)
        current_results = evaluate_model_generative_task(similarity_model, similarity_tokenizer, eval_subset, metric_names=metric_names)
        current_results['elapsed_seconds'] = float(time.time() - started_at)

        STATE['results'][experiment_key] = current_results
        STATE['completed_keys'].append(experiment_key)
        save_state(STATE_PATH, STATE)

        print(f"Finished {experiment_key} in {current_results['elapsed_seconds']:.1f}s")
    finally:
        for hook in hooks:
            hook.remove()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def run_part1_layer_dropping_sweep(metric_names=None):
    if metric_names is None:
        metric_names = SWEEP_METRIC_NAMES

    run_with_hooks('baseline_no_drop', lambda: [], metric_names=metric_names)

    for idx in range(NUM_ENCODER_BLOCKS):
        run_with_hooks(
            f'drop_encoder_block_{idx}',
            lambda idx=idx: [drop_encoder_block(flan_model, idx)],
            metric_names=metric_names
        )

    for idx in range(NUM_DECODER_BLOCKS):
        run_with_hooks(
            f'drop_decoder_block_{idx}',
            lambda idx=idx: [drop_decoder_block(flan_model, idx)],
            metric_names=metric_names
        )

    for end_idx in range(NUM_ENCODER_BLOCKS - 1):
        run_with_hooks(
            f'drop_encoder_block_from_beginning_{end_idx}',
            lambda end_idx=end_idx: [drop_encoder_block(flan_model, i) for i in range(0, end_idx + 1)],
            metric_names=metric_names
        )

    for end_idx in range(NUM_DECODER_BLOCKS - 1):
        run_with_hooks(
            f'drop_decoder_block_from_beginning_{end_idx}',
            lambda end_idx=end_idx: [drop_decoder_block(flan_model, i) for i in range(0, end_idx + 1)],
            metric_names=metric_names
        )

    for start_idx in range(NUM_ENCODER_BLOCKS - 1, 0, -1):
        run_with_hooks(
            f'drop_encoder_block_from_end_{start_idx}',
            lambda start_idx=start_idx: [drop_encoder_block(flan_model, i) for i in range(start_idx, NUM_ENCODER_BLOCKS)],
            metric_names=metric_names
        )

    for start_idx in range(NUM_DECODER_BLOCKS - 1, 0, -1):
        run_with_hooks(
            f'drop_decoder_block_from_end_{start_idx}',
            lambda start_idx=start_idx: [drop_decoder_block(flan_model, i) for i in range(start_idx, NUM_DECODER_BLOCKS)],
            metric_names=metric_names
        )

    print('Completed Part 1 layer-dropping sweep (6 loop groups plus baseline).')
    print('Results keys:', len(STATE['results']))
    print('Saved to:', STATE_PATH)


# Manual call only; this cell no longer starts experiments automatically.
# run_part1_layer_dropping_sweep(metric_names=CONFIG.get('sweep_metric_names', ['similarity']))

In [ ]:
print('Completed experiments:', len(STATE['completed_keys']))

if 'baseline_no_drop' in STATE['results']:
    print('Baseline similarity mean:', STATE['results']['baseline_no_drop']['similarity_mean'])

example_key = next(iter(STATE['results'])) if len(STATE['results']) > 0 else None
if example_key is not None:
    print('Example result key:', example_key)
    print('Example result:', STATE['results'][example_key])

In [ ]:
def build_experiment_registry():
    registry = {}
    groups = {
        'part1_layer_dropping': [],
        'part1_odd_even': [],
        'part1_feed_forward_dropping': [],
        'part2_attention_masking': [],
        'part2_feed_forward_masking': [],
        'part3_gaussian_noise': []
    }

    def add(group_name, key, hook_fn):
        registry[key] = hook_fn
        groups[group_name].append(key)

    add('part1_layer_dropping', 'baseline_no_drop', lambda: [])

    for idx in range(NUM_ENCODER_BLOCKS):
        add('part1_layer_dropping', f'drop_encoder_block_{idx}', lambda idx=idx: [drop_encoder_block(flan_model, idx)])
    for idx in range(NUM_DECODER_BLOCKS):
        add('part1_layer_dropping', f'drop_decoder_block_{idx}', lambda idx=idx: [drop_decoder_block(flan_model, idx)])
    for idx in range(NUM_ENCODER_BLOCKS - 1):
        add('part1_layer_dropping', f'drop_encoder_block_from_beginning_{idx}', lambda idx=idx: [drop_encoder_block(flan_model, i) for i in range(0, idx + 1)])
    for idx in range(NUM_DECODER_BLOCKS - 1):
        add('part1_layer_dropping', f'drop_decoder_block_from_beginning_{idx}', lambda idx=idx: [drop_decoder_block(flan_model, i) for i in range(0, idx + 1)])
    for idx in range(NUM_ENCODER_BLOCKS - 1, 0, -1):
        add('part1_layer_dropping', f'drop_encoder_block_from_end_{idx}', lambda idx=idx: [drop_encoder_block(flan_model, i) for i in range(idx, NUM_ENCODER_BLOCKS)])
    for idx in range(NUM_DECODER_BLOCKS - 1, 0, -1):
        add('part1_layer_dropping', f'drop_decoder_block_from_end_{idx}', lambda idx=idx: [drop_decoder_block(flan_model, i) for i in range(idx, NUM_DECODER_BLOCKS)])

    add('part1_odd_even', 'drop_encoder_block_all_even', lambda: [drop_encoder_block(flan_model, i) for i in range(0, NUM_ENCODER_BLOCKS, 2)])
    add('part1_odd_even', 'drop_decoder_block_all_even', lambda: [drop_decoder_block(flan_model, i) for i in range(0, NUM_DECODER_BLOCKS, 2)])
    add('part1_odd_even', 'drop_encoder_block_all_odd', lambda: [drop_encoder_block(flan_model, i) for i in range(1, NUM_ENCODER_BLOCKS, 2)])
    add('part1_odd_even', 'drop_decoder_block_all_odd', lambda: [drop_decoder_block(flan_model, i) for i in range(1, NUM_DECODER_BLOCKS, 2)])
    add('part1_odd_even', 'drop_blocks_all_even', lambda: [drop_encoder_block(flan_model, i) for i in range(0, NUM_ENCODER_BLOCKS, 2)] + [drop_decoder_block(flan_model, i) for i in range(0, NUM_DECODER_BLOCKS, 2)])
    add('part1_odd_even', 'drop_blocks_all_odd', lambda: [drop_encoder_block(flan_model, i) for i in range(1, NUM_ENCODER_BLOCKS, 2)] + [drop_decoder_block(flan_model, i) for i in range(1, NUM_DECODER_BLOCKS, 2)])

    add('part1_feed_forward_dropping', 'drop_feed_forward_encoder_all_even', lambda: [drop_encoder_block_feed_forward_layer(flan_model, i) for i in range(0, NUM_ENCODER_BLOCKS, 2)])
    add('part1_feed_forward_dropping', 'drop_feed_forward_decoder_all_even', lambda: [drop_decoder_block_feed_forward_layer(flan_model, i) for i in range(0, NUM_DECODER_BLOCKS, 2)])
    add('part1_feed_forward_dropping', 'drop_feed_forward_encoder_all_odd', lambda: [drop_encoder_block_feed_forward_layer(flan_model, i) for i in range(1, NUM_ENCODER_BLOCKS, 2)])
    add('part1_feed_forward_dropping', 'drop_feed_forward_decoder_all_odd', lambda: [drop_decoder_block_feed_forward_layer(flan_model, i) for i in range(1, NUM_DECODER_BLOCKS, 2)])
    add('part1_feed_forward_dropping', 'drop_feed_forward_all_even', lambda: [drop_encoder_block_feed_forward_layer(flan_model, i) for i in range(0, NUM_ENCODER_BLOCKS, 2)] + [drop_decoder_block_feed_forward_layer(flan_model, i) for i in range(0, NUM_DECODER_BLOCKS, 2)])
    add('part1_feed_forward_dropping', 'drop_feed_forward_all_odd', lambda: [drop_encoder_block_feed_forward_layer(flan_model, i) for i in range(1, NUM_ENCODER_BLOCKS, 2)] + [drop_decoder_block_feed_forward_layer(flan_model, i) for i in range(1, NUM_DECODER_BLOCKS, 2)])
    add('part1_feed_forward_dropping', 'drop_feed_forward_encoder_all', lambda: [drop_encoder_block_feed_forward_layer(flan_model, i) for i in range(NUM_ENCODER_BLOCKS)])
    add('part1_feed_forward_dropping', 'drop_feed_forward_decoder_all', lambda: [drop_decoder_block_feed_forward_layer(flan_model, i) for i in range(NUM_DECODER_BLOCKS)])
    add('part1_feed_forward_dropping', 'drop_feed_forward_all', lambda: [drop_encoder_block_feed_forward_layer(flan_model, i) for i in range(NUM_ENCODER_BLOCKS)] + [drop_decoder_block_feed_forward_layer(flan_model, i) for i in range(NUM_DECODER_BLOCKS)])

    for threshold in [0.1, 0.25, 0.5, 0.75]:
        add('part2_attention_masking', f'mask_encoder_all_self_attention_{threshold}', lambda threshold=threshold: [mask_self_attention_heads_encoder_block(flan_model, i, threshold) for i in range(NUM_ENCODER_BLOCKS)])
        add('part2_attention_masking', f'mask_decoder_all_self_attention_{threshold}', lambda threshold=threshold: [mask_self_attention_heads_decoder_block(flan_model, i, threshold) for i in range(NUM_DECODER_BLOCKS)])
        add('part2_attention_masking', f'mask_decoder_all_cross_attention_{threshold}', lambda threshold=threshold: [mask_cross_attention_heads_decoder_block(flan_model, i, threshold) for i in range(NUM_DECODER_BLOCKS)])
        add('part2_attention_masking', f'mask_decoder_all_self_cross_attention_{threshold}', lambda threshold=threshold: [hook for i in range(NUM_DECODER_BLOCKS) for hook in (mask_self_attention_heads_decoder_block(flan_model, i, threshold), mask_cross_attention_heads_decoder_block(flan_model, i, threshold))])
        add('part2_attention_masking', f'mask_encoder_decoder_all_self_attention_{threshold}', lambda threshold=threshold: [mask_self_attention_heads_encoder_block(flan_model, i, threshold) for i in range(NUM_ENCODER_BLOCKS)] + [mask_self_attention_heads_decoder_block(flan_model, i, threshold) for i in range(NUM_DECODER_BLOCKS)])
        add('part2_attention_masking', f'mask_encoder_decoder_all_self_cross_attention_{threshold}', lambda threshold=threshold: [mask_self_attention_heads_encoder_block(flan_model, i, threshold) for i in range(NUM_ENCODER_BLOCKS)] + [hook for i in range(NUM_DECODER_BLOCKS) for hook in (mask_self_attention_heads_decoder_block(flan_model, i, threshold), mask_cross_attention_heads_decoder_block(flan_model, i, threshold))])

    for threshold in [0.1, 0.2, 0.25, 0.5, 0.75, 0.8]:
        add('part2_feed_forward_masking', f'mask_encoder_feed_forward_all_{threshold}', lambda threshold=threshold: [mask_encoder_block_feed_forward_layer(flan_model, i, threshold) for i in range(NUM_ENCODER_BLOCKS)])
        add('part2_feed_forward_masking', f'mask_decoder_feed_forward_all_{threshold}', lambda threshold=threshold: [mask_decoder_block_feed_forward_layer(flan_model, i, threshold) for i in range(NUM_DECODER_BLOCKS)])
        add('part2_feed_forward_masking', f'mask_feed_forward_all_{threshold}', lambda threshold=threshold: [mask_encoder_block_feed_forward_layer(flan_model, i, threshold) for i in range(NUM_ENCODER_BLOCKS)] + [mask_decoder_block_feed_forward_layer(flan_model, i, threshold) for i in range(NUM_DECODER_BLOCKS)])

    for sd in [0.01, 0.05, 0.1, 0.25, 0.5, 1.0]:
        add('part3_gaussian_noise', f'add_gaussian_noise_encoder_all_{sd}', lambda sd=sd: [add_gaussian_noise_encoder_block(flan_model, i, standard_deviation=sd, mean=0.0) for i in range(NUM_ENCODER_BLOCKS)])
        add('part3_gaussian_noise', f'add_gaussian_noise_decoder_all_{sd}', lambda sd=sd: [add_gaussian_noise_decoder_block(flan_model, i, standard_deviation=sd, mean=0.0) for i in range(NUM_DECODER_BLOCKS)])
        add('part3_gaussian_noise', f'add_gaussian_noise_all_{sd}', lambda sd=sd: [add_gaussian_noise_encoder_block(flan_model, i, standard_deviation=sd, mean=0.0) for i in range(NUM_ENCODER_BLOCKS)] + [add_gaussian_noise_decoder_block(flan_model, i, standard_deviation=sd, mean=0.0) for i in range(NUM_DECODER_BLOCKS)])
        add('part3_gaussian_noise', f'add_gaussian_noise_encoder_feed_forward_all_{sd}', lambda sd=sd: [add_gaussian_noise_encoder_block_feed_forward_layer(flan_model, i, standard_deviation=sd, mean=0.0) for i in range(NUM_ENCODER_BLOCKS)])
        add('part3_gaussian_noise', f'add_gaussian_noise_decoder_feed_forward_all_{sd}', lambda sd=sd: [add_gaussian_noise_decoder_block_feed_forward_layer(flan_model, i, standard_deviation=sd, mean=0.0) for i in range(NUM_DECODER_BLOCKS)])
        add('part3_gaussian_noise', f'add_gaussian_noise_feed_forward_all_{sd}', lambda sd=sd: [add_gaussian_noise_encoder_block_feed_forward_layer(flan_model, i, standard_deviation=sd, mean=0.0) for i in range(NUM_ENCODER_BLOCKS)] + [add_gaussian_noise_decoder_block_feed_forward_layer(flan_model, i, standard_deviation=sd, mean=0.0) for i in range(NUM_DECODER_BLOCKS)])

    return registry, groups


EXPERIMENT_REGISTRY, EXPERIMENT_GROUPS = build_experiment_registry()
print('Registered experiments:', len(EXPERIMENT_REGISTRY))
for group_name, keys in EXPERIMENT_GROUPS.items():
    print(group_name, len(keys))


def result_has_metrics(result, metric_names):
    if not isinstance(result, dict):
        return False
    if 'similarity' in metric_names and 'similarity_mean' not in result:
        return False
    if 'rouge' in metric_names and 'rouge' not in result:
        return False
    if 'meteor' in metric_names and 'meteor_mean' not in result:
        return False
    return True


def execute_registered_experiment(experiment_key, metric_names=None, force=False):
    if metric_names is None:
        metric_names = CONFIG.get('full_metric_names', ['similarity', 'rouge', 'meteor'])

    force = force or bool(CONFIG.get('rewrite_existing_metrics', False))

    if experiment_key not in EXPERIMENT_REGISTRY:
        print(f'Skipping {experiment_key}: no registered hook function')
        return

    existing_result = STATE['results'].get(experiment_key, {})
    if (not force) and result_has_metrics(existing_result, metric_names):
        print(f'Skipping {experiment_key}: requested metrics already present')
        return

    hooks = []
    started_at = time.time()
    try:
        hooks = EXPERIMENT_REGISTRY[experiment_key]()
        eval_subset = obtain_model_generated_outputs_subset(flan_model, flan_tokenizer, self_instruct_subset)
        new_result = evaluate_model_generative_task(similarity_model, similarity_tokenizer, eval_subset, metric_names=metric_names)
        merged_result = dict(existing_result)
        merged_result.update(new_result)
        merged_result['elapsed_seconds'] = float(time.time() - started_at)
        merged_result['last_metric_update_unix'] = time.time()

        STATE['results'][experiment_key] = merged_result
        if experiment_key not in STATE['completed_keys']:
            STATE['completed_keys'].append(experiment_key)
        save_state(STATE_PATH, STATE)
        print(f"Finished {experiment_key} with metrics {metric_names} in {merged_result['elapsed_seconds']:.1f}s")
    finally:
        for hook in hooks:
            hook.remove()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def backfill_missing_full_metrics(metric_names=None, keys=None):
    if metric_names is None:
        metric_names = CONFIG.get('full_metric_names', ['similarity', 'rouge', 'meteor'])
    if keys is None:
        keys = list(STATE.get('completed_keys', []))

    for key in keys:
        execute_registered_experiment(key, metric_names=metric_names, force=False)

    print('Backfill finished.')


def rewrite_completed_metrics_for_group(group_name, metric_names=None):
    if metric_names is None:
        metric_names = CONFIG.get('full_metric_names', ['similarity', 'rouge', 'meteor'])

    group_keys = EXPERIMENT_GROUPS.get(group_name, [])
    completed_or_saved_keys = set(STATE.get('completed_keys', [])) | set(STATE.get('results', {}).keys())
    keys_to_rewrite = [key for key in group_keys if key in completed_or_saved_keys]

    print(f'Rewriting {len(keys_to_rewrite)} already-run experiments from group {group_name}')
    for key in keys_to_rewrite:
        execute_registered_experiment(key, metric_names=metric_names, force=True)

    print(f'Rewrite finished for group {group_name}.')


def rewrite_part3_gaussian_metrics(metric_names=None):
    rewrite_completed_metrics_for_group('part3_gaussian_noise', metric_names=metric_names)

In [ ]:
def run_experiment_group(group_name, metric_names=None, force=False):
    if metric_names is None:
        metric_names = CONFIG.get('sweep_metric_names', ['similarity'])
    keys = EXPERIMENT_GROUPS[group_name]
    print(f'Running group {group_name} with {len(keys)} experiments and metrics {metric_names}')
    for key in keys:
        execute_registered_experiment(key, metric_names=metric_names, force=force)


def run_part1_full_suite(metric_names=None, force=False):
    run_experiment_group('part1_layer_dropping', metric_names=metric_names, force=force)
    run_experiment_group('part1_odd_even', metric_names=metric_names, force=force)
    run_experiment_group('part1_feed_forward_dropping', metric_names=metric_names, force=force)


def run_part2_full_suite(metric_names=None, force=False):
    run_experiment_group('part2_attention_masking', metric_names=metric_names, force=force)
    run_experiment_group('part2_feed_forward_masking', metric_names=metric_names, force=force)


def run_part3_full_suite(metric_names=None, force=False):
    run_experiment_group('part3_gaussian_noise', metric_names=metric_names, force=force)


def run_all_parts_full_suite(metric_names=None, force=False):
    run_part1_full_suite(metric_names=metric_names, force=force)
    run_part2_full_suite(metric_names=metric_names, force=force)
    run_part3_full_suite(metric_names=metric_names, force=force)


# Use these manually depending on what you want to run.
# Similarity-only sweep (fast):
# run_all_parts_full_suite(metric_names=CONFIG.get('sweep_metric_names', ['similarity']))

# Append missing full metrics to already-completed similarity-only results:
# backfill_missing_full_metrics(metric_names=CONFIG.get('full_metric_names', ['similarity', 'rouge', 'meteor']))

# Full metrics for all Part 1/2/3 experiments (slow):
run_all_parts_full_suite(metric_names=CONFIG.get('full_metric_names', ['similarity', 'rouge', 'meteor']))
# rewrite_part3_gaussian_metrics(
#     metric_names=CONFIG.get('full_metric_names', ['similarity', 'rouge', 'meteor'])
# )

In [ ]:
expected_keys = ['baseline_no_drop']

expected_keys += [f'drop_encoder_block_{idx}' for idx in range(NUM_ENCODER_BLOCKS)]
expected_keys += [f'drop_decoder_block_{idx}' for idx in range(NUM_DECODER_BLOCKS)]
expected_keys += [f'drop_encoder_block_from_beginning_{end_idx}' for end_idx in range(NUM_ENCODER_BLOCKS - 1)]
expected_keys += [f'drop_decoder_block_from_beginning_{end_idx}' for end_idx in range(NUM_DECODER_BLOCKS - 1)]
expected_keys += [f'drop_encoder_block_from_end_{start_idx}' for start_idx in range(NUM_ENCODER_BLOCKS - 1, 0, -1)]
expected_keys += [f'drop_decoder_block_from_end_{start_idx}' for start_idx in range(NUM_DECODER_BLOCKS - 1, 0, -1)]

expected_set = set(expected_keys)
completed_set = set(STATE.get('completed_keys', []))

completed_in_plan = sorted(expected_set.intersection(completed_set))
pending_in_plan = sorted(expected_set.difference(completed_set))

elapsed_values = []
for key in completed_in_plan:
    key_result = STATE.get('results', {}).get(key, {})
    elapsed = key_result.get('elapsed_seconds', None)
    if isinstance(elapsed, (int, float)) and elapsed > 0:
        elapsed_values.append(float(elapsed))

print('Total planned experiments:', len(expected_keys))
print('Completed in plan:', len(completed_in_plan))
print('Pending in plan:', len(pending_in_plan))

if len(elapsed_values) > 0:
    avg_seconds = sum(elapsed_values) / len(elapsed_values)
    eta_seconds = avg_seconds * len(pending_in_plan)
    print(f'Average seconds per completed experiment: {avg_seconds:.1f}s')
    print(f'Estimated remaining time: {eta_seconds / 60:.1f} min ({eta_seconds / 3600:.2f} h)')
else:
    print('Not enough timing data yet for ETA (run at least one completed hook).')

if len(pending_in_plan) > 0:
    print('Next pending key:', pending_in_plan[0])